In [ ]:
from sheet_1 import simulate
from qiskit import QuantumCircuit
import numpy as np




# Gates

Qiskit can transpile all gates to a series of basis gates, in our case the one qubit rotation gate 'u' and the two qubit gate 'cx'. These two gates were defined in 'apply_u_on_state.py' and 'apply_cx_on_state.py'. 

## u gate

In [ ]:
import numpy as np

def apply_u_on_state(state: np.ndarray, u: np.ndarray, acting_on: int) -> np.ndarray:
    """Applies a single-qubit gate to the given state vector. The gate is defined 
    by the 2x2 matrix u, and it acts on the qubit specified by acting_on"""
    number_of_qubits=state.ndim
    acting_on=number_of_qubits-acting_on-1
    old_indices = [i for i in range(number_of_qubits)]
    new_indices = old_indices.copy()
    new_indices[acting_on] = 51
    result=np.einsum(u,[51,acting_on],state,old_indices,new_indices)
    return result

## cx gate

In [ ]:
import numpy as np

def apply_cx_on_state(state: np.ndarray, cx: np.ndarray, acting_on1: int, acting_on2: int) -> np.ndarray:
    '''Applies a CNOT gate to the given state vector. The CNOT gate is defined 
    by the 4x4 matrix cx, and it acts on the qubits specified by acting_on1 (control) 
    and acting_on2 (target). The state vector is modified in-place.'''
    number_of_qubits=state.ndim
    acting_on1=number_of_qubits-1-acting_on1
    acting_on2=number_of_qubits-1-acting_on2
    cx_matrix=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    cx = np.reshape(cx_matrix,(2,2,2,2))
    old_indices = [i for i in range(number_of_qubits)]
    new_indices = old_indices.copy()
    new_indices[acting_on1] = 51
    new_indices[acting_on2] = 50
    result=np.einsum(cx,[51,50,acting_on1,acting_on2],state,old_indices,new_indices)
    return result

# Simulator

The simulator uses the two gates 'u' and 'cx'. It simulates the, to basisgates 'u' and 'cx' transpiled, quantum circuit and returns the final state vector.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UGate, CXGate
from qiskit_aer import AerSimulator
from sheet_1.apply_cx_on_state import apply_cx_on_state
from sheet_1.apply_u_on_state import apply_u_on_state
import numpy as np
from qiskit.quantum_info import Statevector



def simulate(qc: QuantumCircuit, parameters: dict | None = None) -> np.ndarray:
    """Simulates the given quantum circuit and returns the final state vector."""
    number_of_qubits=qc.num_qubits
    psi=np.zeros(2**number_of_qubits)
    psi[0]=1
    psi_converted = np.reshape(psi,[2]*number_of_qubits)

    for instr, qargs, cargs in qc.data:
        qubit_indices = [qc.find_bit(q).index for q in qargs]

        if instr.name == 'cx':
            cx=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
            psi_converted=apply_cx_on_state(psi_converted,cx,qubit_indices[0], qubit_indices[1])
        elif instr.name == 'u':
            u=UGate(instr.params[0],instr.params[1],instr.params[2]).to_matrix()
            psi_converted=apply_u_on_state(psi_converted,u,qubit_indices[0])
        else:
            continue
    
    psi_final = np.reshape(np.asarray(psi_converted), (2**number_of_qubits,))
    return psi_final












The simulator was tested with a simple and a random circuit. 

# Benchmarks

The runtime of our own simulator and the Aer_simulator were compared. To do this two functions constructing circuits and parameters for both simulators were defined. Then benchmarked testfunctions were defined. In a last step the two statevectors were compared, to ensure correctness of the simulator. This test is not benchmarked.  

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.random import random_circuit
from sheet_1 import quantum_simulator
from qiskit_aer import AerSimulator

import numpy as np
import pytest


# ----------------------------
# Circuit construction helpers
# ----------------------------

def construct_simple_circuit():
    number_of_qubits = 3
    qc = QuantumCircuit(number_of_qubits)
    qc.h(0)
    qc.h(1)
    qc.h(2)
    return qc


def construct_random_circuit():
    # fixed seed for reproducibility
    return random_circuit(num_qubits=12, depth=10, measure=False, seed=42)


# ----------------------------
# Shared parametrization
# ----------------------------

@pytest.fixture(params=[construct_simple_circuit, construct_random_circuit])
def circuit(request):
    qc = request.param()
    qc = transpile(qc, basis_gates=['u', 'cx'])
    return qc


# ----------------------------
# Benchmarks
# ----------------------------

def test_aer_simulation(benchmark, circuit):
    """Benchmark Qiskit Aer statevector simulation"""
    circuit.save_statevector()
    simulator = AerSimulator(method="statevector")
    compiled = transpile(circuit, simulator)

    def run():
        result = simulator.run(compiled).result()
        return result.get_statevector()

    result = benchmark(run)

    # basic sanity check (not strict comparison here)
    assert result is not None



def test_own_simulation(benchmark, circuit):
    """Benchmark your custom simulator"""
    def run():
        return quantum_simulator.simulate(circuit, None)

    result = benchmark(run)

    assert result is not None


# ----------------------------
# Correctness test (no benchmark)
# ----------------------------

@pytest.mark.parametrize("construction_function", [construct_simple_circuit, construct_random_circuit])
def test_correctness(construction_function):
    qc = construction_function()
    qc = transpile(qc, basis_gates=['u', 'cx'])

    aer_state = Statevector.from_instruction(qc)
    own_state = quantum_simulator.simulate(qc, None)

    # compare states up to global phase
    idx = np.argmax(np.abs(aer_state))
    global_phase = aer_state.data[idx] / own_state[idx]

    assert np.allclose(aer_state, own_state * global_phase)

# Gates using numba

The code for the gates was modified using numba to substitude 'np.einsum' with for-loops.



## 'u' gate

In [ ]:
import numba 
import numpy as np

@numba.jit(cache=True)
def apply_u_numba(state: np.ndarray, u: np.ndarray, acting_on: int) -> np.ndarray:
    number_of_qubits=int(np.log2(state.size))
    #acting_on=number_of_qubits-acting_on-1
    for idx_lower in range(0, 2**acting_on):
        for idx_upper in range(0, 2**number_of_qubits, 2**(acting_on+1)):
            idx0 = idx_lower + idx_upper
            idx1 = idx_upper + idx_lower + 2**acting_on
            s0 = state[idx0]
            s1 = state[idx1]
            state[idx0] = u[0, 0] * s0 + u[0, 1] * s1
            state[idx1] = u[1, 0] * s0 + u[1, 1] * s1
    return state

## 'cx' Gate

In [ ]:
#code!!!!!!!

## Test for numba gates

The gates with numba were tested against the AerSimulator. 


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UGate, CXGate
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np
from qiskit.circuit.random import random_circuit
from sheet_1.apply_u_numba import apply_u_numba

theta = np.pi/3
phi=np.pi/8
lam=0
apply_to=4
number_of_qubits=9

qc = random_circuit(num_qubits=number_of_qubits, depth=10, measure=False, seed=42)
tqc = transpile(qc, basis_gates=['u', 'cx'])
initial_state = Statevector.from_instruction(tqc)

# simulate result using AerSimulator
qc2 = QuantumCircuit(number_of_qubits)
qc2.u(theta,phi,lam,apply_to)
Aer_result = initial_state.evolve(qc2)

# set up u gate
u=UGate(theta,phi,lam).to_matrix()

result = apply_u_numba(initial_state.data, u, apply_to)


global_phase=1

assert np.allclose(Aer_result.data, result*global_phase, atol=1e-12)


## Simulator with numba

The quantum simulator was adjusted to use the gates defined with numba. The functions are apply_u_numba and apply_cx_numba. !!!!!!!!!!!!!!! change cx gate to numba

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UGate, CXGate
from qiskit_aer import AerSimulator
from sheet_1.apply_cx_on_state import apply_cx_on_state
from sheet_1.apply_u_numba import apply_u_numba
'''u gate using numba'''
import numpy as np
import numba
from qiskit.quantum_info import Statevector

#change apply_cx_on_state to numba version

@numba.jit(cache=True)
def simulate_numba(qc: QuantumCircuit, parameters: dict | None = None) -> np.ndarray:
    """Simulates the given quantum circuit and returns the final state vector."""
    number_of_qubits=qc.num_qubits
    psi=np.zeros(2**number_of_qubits)
    psi[0]=1
    psi_converted = np.reshape(psi,[2]*number_of_qubits)

    for instr, qargs, cargs in qc.data:
        qubit_indices = [qc.find_bit(q).index for q in qargs]

        if instr.name == 'cx':
            cx=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
            psi_converted=apply_cx_on_state(psi_converted,cx,qubit_indices[0], qubit_indices[1])
        elif instr.name == 'u':
            u=UGate(instr.params[0],instr.params[1],instr.params[2]).to_matrix()
            psi_converted=apply_u_numba(psi_converted,u,qubit_indices[0])
        else:
            continue
    
    psi_final = np.reshape(np.asarray(psi_converted), (2**number_of_qubits,))
    return psi_final

            









## Benchmark numba

The simulator using numba was bechmarked against the AerSimulator. 

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.random import random_circuit
from sheet_1 import simulator_numba
from qiskit_aer import AerSimulator

import numpy as np
import pytest


# ----------------------------
# Circuit construction helpers
# ----------------------------

def construct_simple_circuit():
    number_of_qubits = 3
    qc = QuantumCircuit(number_of_qubits)
    qc.h(0)
    qc.h(1)
    qc.h(2)
    return qc


def construct_random_circuit():
    # fixed seed for reproducibility
    return random_circuit(num_qubits=12, depth=10, measure=False, seed=42)


# ----------------------------
# Shared parametrization
# ----------------------------

@pytest.fixture(params=[construct_simple_circuit, construct_random_circuit])
def circuit(request):
    qc = request.param()
    qc = transpile(qc, basis_gates=['u', 'cx'])
    return qc


# ----------------------------
# Benchmarks
# ----------------------------

# sollte gleich bleiben, oder?

def test_aer_simulation(benchmark, circuit):
    """Benchmark Qiskit Aer statevector simulation"""
    circuit.save_statevector()
    simulator = AerSimulator(method="statevector")
    compiled = transpile(circuit, simulator)

    def run():
        result = simulator.run(compiled).result()
        return result.get_statevector()

    result = benchmark(run)

    # basic sanity check (not strict comparison here)
    assert result is not None



def test_own_simulation(benchmark, circuit):
    """Benchmark your custom simulator"""
    def run():
        return simulator_numba.simulate_numba(circuit, None)

    result = benchmark(run)

    assert result is not None


# ----------------------------
# Correctness test (no benchmark)
# ----------------------------

@pytest.mark.parametrize("construction_function", [construct_simple_circuit, construct_random_circuit])
def test_correctness(construction_function):
    qc = construction_function()
    qc = transpile(qc, basis_gates=['u', 'cx'])

    aer_state = Statevector.from_instruction(qc)
    own_state = simulator_numba.simulate_numba(qc, None)

    # compare states up to global phase
    idx = np.argmax(np.abs(aer_state))
    global_phase = aer_state.data[idx] / own_state[idx]

    assert np.allclose(aer_state, own_state * global_phase)